## Prepare the environment

In keeping with good DevOps practices, we will deploy our infrastructure - starting with the Kubernetes cluster - using infrastructure-as-code and configuration-as-code principles:

-   The process of provisioning and deploying the infrastructure is scalable, because it is heavily automated. It is easy to rebuild the system, without requiring effort or expertise.
-   Everything needed to deploy the infrastructure is in version control.
-   The infrastructure is immutable - no manual updates or changes.

We will use two IaC/CaC tools to prepare our Kubernetes cluster:

-   [Terraform](https://www.terraform.io/), which we’ll use to provision the resources on our cloud infrastructure provider.
-   [Ansible](https://github.com/ansible/ansible), which we’ll use to configure and deploy Kubernetes, and then to set up the Kubernetes cluster and the services running on it.

both of which are aligned with the principles above.

In this notebook, which will run in the Chameleon Jupyter environment, we will install and configure Terraform in our environment, and in a later notebook, we will install and configure Ansible in this same environment. This is a *one-time* step that an engineer would ordinarily do just once, when “onboarding”, on their own computer.

> **Note**: This is a Bash notebook, so you will run it with a Bash kernel. You can change the kernel (if needed) by clicking the kernel name in the top right of the Jupyter interface.

### Get infrastructure configuration

Following IaC principles, our infrastructure configuration is all in version control! We have organized all of the materials that “describe” the deployment in our “IaC repository”: <https://github.com/teaching-on-testbeds/gourmetgram-iac.git>.

This repository has the following structure:

    ├── tf
    │   └── kvm
    ├── ansible
    │   ├── general
    │   ├── pre_k8s
    │   ├── k8s
    │   ├── post_k8s
    │   └── argocd
    ├── k8s
    │   ├── platform
    │   ├── staging
    │   ├── canary
    │   └── production
    └── workflows

-   The `tf` directory includes materials needed for Terraform to provision resources from the cloud provider. This is a “Day 0” setup task.
-   The “Day 1” setup task is to install and configure Kubernetes on the resources. We use Ansible, and the materials are in the `ansible` directory in the `pre_k8s`, `k8s` and `post_k8s` subdirectories. (The `general` directory is just for learning.)
-   The applications that we will be deployed in Kubernetes are defined in the `k8s` directory:
    -   `platform` has all the “accessory” services we need to support our machine learning application. In this example, it has a model registry (where we save trained model artifacts after they are “built”) and the associated database and object store services used by the model registry. It also has a gateway service that routes some traffic to a `canary` environment and the rest to `production`. More generally, “platform” may include experiment tracking, evaluation and monitoring, and other related services.
    -   `staging`, `canary`, and `production` are deployments of our GourmetGram application. A new model or application version starts off in `staging`; after some internal tests it may be promoted to `canary` where it is served to some live users; and after further evaluation and monitoring, it may be promoted to `production`.
-   We use Ansible to “register” these applications in ArgoCD, using the playbooks in the `ansible/argocd` directory. ArgoCD is a continuous delivery tool for Kubernetes that automatically deploys and updates applications based on the latest version of its manifests.
-   From “Day 2” and on, during the lifecycle of the application, we use ArgoCD and Argo Workflows to handle model and application versions, using the pipelines in `workflows`.

In the next cell, we get a copy of the [GourmetGram infrastructure repository](https://github.com/teaching-on-testbeds/gourmetgram-iac.git):

In [ ]:
# runs in Chameleon Jupyter environment
git clone --recurse-submodules https://github.com/teaching-on-testbeds/gourmetgram-iac.git /work/gourmetgram-iac

Note that we use the `--recurse-submodules` argument to `git clone` - we are including Kubespray, an Ansible-based project for deploying Kubernetes, inside our IaC repository as a submodule.

Among the automation and CI/CD tools mentioned above:

-   Terraform and Ansible run on the engineer’s own computer, and communicate with the cloud provider/cloud resources over a network.
-   ArgoCD and Argo Workflows run on the cloud resources themselves.

So, a necessary prerequisite for this workflow is to download, install, and configure Terraform and Ansible on “our own computer” - except in this case, we will use the Chameleon Jupyter environment as “our computer”.

### Install and configure Terraform

Before we can use Terraform, we’ll need to download a Terraform client. The following cell will download the Terraform client and “install” it in this environment:

In [ ]:
# runs in Chameleon Jupyter environment
mkdir -p /work/.local/bin
wget https://releases.hashicorp.com/terraform/1.14.4/terraform_1.14.4_linux_amd64.zip
unzip -o -q terraform_1.14.4_linux_amd64.zip
mv terraform /work/.local/bin
rm terraform_1.14.4_linux_amd64.zip

The Terraform client has been installed to: `/work/.local/bin`. In order to run `terraform` commands, we will have to add this directory to our `PATH`, which tells the system where to look for executable files.

In [ ]:
# runs in Chameleon Jupyter environment
export PATH=/work/.local/bin:$PATH

Let’s make sure we can now run `terraform` commands. The following cell should print usage information for the `terraform` command, since we run it without any subcommands:

In [ ]:
# runs in Chameleon Jupyter environment
terraform

Terraform works by communicating with a cloud provider (either a commercial cloud, like AWS or GCP, or a private cloud, like an on-premises OpenStack cloud, or a hybrid cloud with both types of resources). We will need to prepare credentials with which it can act on our behalf on the Chameleon OpenStack cloud. This is a one-time procedure.

To get credentials, open the Horizon GUI:

-   from the Chameleon website
-   click “Experiment” \> “KVM@TACC”
-   log in if prompted to do so
-   check the project drop-down menu near the top left (which shows e.g. “CHI-XXXXXX”), and make sure the correct project is selected.

On the left side, expand the “Identity” section and click on “Application Credentials”. Then, click “Create Application Credential”.

-   In the “Name”, field, use “mlops-lab”.
-   Set the “Expiration” date and time to the due date of this lab. (Note that this will be in UTC time, not your local time zone.) This ensures that if your credential is leaked (e.g. you accidentially push it to a public Github repository), the damage is mitigated.
-   Click “Create Application Credential”.
-   Choose “Download clouds.yaml”.

The `clouds.yaml` file will look something like this (expect with an alphanumeric string in place of `REDACTED_UNIQUE_ID` and `REDACTED_SECRET`):

    clouds:
      openstack:
        auth:
          auth_url: https://kvm.tacc.chameleoncloud.org:5000
          application_credential_id: "REDACTED_UNIQUE_ID"
          application_credential_secret: "REDACTED_SECRET"
        region_name: "KVM@TACC"
        interface: "public"
        identity_api_version: 3
        auth_type: "v3applicationcredential"

It lists one or more clouds - in this case, a single cloud named “openstack”, and then for each cloud, specifies how to connect and authenticate to that cloud. In particular, the `application_credential_id` and `application_credential_secret` allow an application like Terraform to interact with the Chameleon cloud on your behalf, without having to use your personal Chameleon login.

Then, in our Terraform configuration, we will have a block like

    provider "openstack" {
      cloud = "openstack"
    }

where the value assigned to `cloud` tells Terraform which cloud in the `clouds.yaml` file to authenticate to.

One nice feature of Terraform is that we can use it to provision resource on multiple clouds. For example, if we wanted to provision resources on both KVM@TACC and CHI@UC (e.g. the training resources on CHI@UC and everything else on KVM@TACC), we might generate application credentials on both sites, and combine them into a `clouds.yaml` like this:

    clouds:
      kvm:
        auth:
          auth_url: https://kvm.tacc.chameleoncloud.org:5000
          application_credential_id: "REDACTED_UNIQUE_ID_KVM"
          application_credential_secret: "REDACTED_SECRET_KVM"
        region_name: "KVM@TACC"
        interface: "public"
        identity_api_version: 3
        auth_type: "v3applicationcredential"
      uc:
        auth:
          auth_url: https://chi.uc.chameleoncloud.org:5000
          application_credential_id: "REDACTED_UNIQUE_ID_UC"
          application_credential_secret: "REDACTED_SECRET_UC"
        region_name: "CHI@UC"
        interface: "public"
        identity_api_version: 3
        auth_type: "v3applicationcredential"

and then in our Terraform configuration, we could multiple OpenStack clouds to use, e.g. have

    provider "openstack" {
      alias = "kvm"
      cloud = "kvm"
    }

    provider "openstack" {
      alias = "uc"
      cloud = "uc"
    }

and in resource definitions, either

    provider = openstack.kvm

or

    provider = openstack.uc

to indicate which cloud to provision on.

For now, since we are just using one cloud, we will leave our `clouds.yaml` as is.

In the file browser in the Chameleon Jupyter environment, you will see a template `clouds.yaml`. Use the file browser to open it, and paste in the

          application_credential_id: "REDACTED_UNIQUE_ID"
          application_credential_secret: "REDACTED_SECRET"

lines from the `clouds.yaml` that you just downloaded from the KVM@TACC GUI (so that it has the “real” credentials in it). Save the file.

Terraform will look for the `clouds.yaml` in either `~/.config/openstack` or the directory from which we run `terraform` - we will move it to the latter directory:

In [ ]:
# runs in Chameleon Jupyter environment
cp clouds.yaml /work/gourmetgram-iac/tf/kvm/clouds.yaml

The Terraform executable has been installed to a location that is not the system-wide location for executable files: `/work/.local/bin`. In order to run `terraform` commands, we will have to add this directory to our `PATH`, which tells the system where to look for executable files.

In [ ]:
# runs in Chameleon Jupyter environment
export PATH=/work/.local/bin:$PATH

and, we’ll have to do that in *each new Bash session*.